# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MotazSameh/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1 — Staleness

I expect older pages to be more likely to need a refresh. I will check whether the declining rate changes across buckets of `days_since_last_update`.

The signal is linked to the refresh decision because staleness is one of the signals behind refresh flags.

I will use the observed bucket rates to decide whether the signal is CONFIRMED, OPPOSITE, MIXED, or FALSE.


In [2]:
# ============================================================
# Signal 1: Staleness
# ============================================================

import pandas as pd
import numpy as np

# Load starter data
df = pd.read_csv("content_refresh_anonymized.csv")

# Create the target
df["is_declining"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("is_declining", "size"),
          declining_rate=("is_declining", "mean")
      )
)

staleness_check["declining_rate"] = (
    staleness_check["declining_rate"] * 100
).round(2)

print("Staleness signal audit")
display(staleness_check)

Staleness signal audit


,n,declining_rate
staleness_bucket,,
0-30 days,20480,51.14
31-90 days,175,58.86
91-180 days,9171,61.11
181+ days,174,47.13


**Verdict: MIXED**

The declining rate increases from 51.14% for pages updated within 30 days to 61.11% for pages in the 91–180 day bucket. However, the 181+ day bucket drops to 47.13%, so the relationship is not consistently increasing with staleness.

The 181+ bucket is also small (n=174), so I would not treat its rate as a strong signal on its own. Overall, staleness looks directionally useful up to 180 days, but the evidence is mixed.


### Signal 2 — Visibility

I expect pages with more search visibility to be more useful refresh candidates because a refresh action has more potential value when a page is already receiving impressions.

I will check whether the declining rate changes across `impressions_90d` buckets.

The result will be treated as an observed directional signal, not as proof that impressions cause a page to decline.


In [3]:
# ============================================================
# Signal 2: Visibility
# ============================================================

# Create visibility buckets
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 100, 500, 1000, np.inf],
    labels=["0-100", "101-500", "501-1000", "1000+"]
)

visibility_check = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("is_declining", "size"),
          declining_rate=("is_declining", "mean")
      )
)

visibility_check["declining_rate"] = (
    visibility_check["declining_rate"] * 100
).round(2)

print("Visibility signal audit")
display(visibility_check)

Visibility signal audit


,n,declining_rate
visibility_bucket,,
0-100,8006,38.92
101-500,5279,60.43
501-1000,3206,60.04
1000+,13509,59.45


**Verdict: CONFIRMED**

The declining rate is much lower for pages with 0–100 impressions (38.92%) and increases to around 60% once impressions exceed 100. The rate remains relatively stable across the higher-volume buckets.

This supports visibility as a useful directional signal for prioritization. It does not show that impressions cause decline; it only shows an observed relationship in this dataset.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### My baseline rule

I will prioritize pages that are both visible and stale.

A page receives a refresh score when it has more than 100 impressions in the last 90 days and has not been updated for at least 91 days. Among these pages, higher impressions receive a higher score because the page has more observed visibility.

The rule produces one reason code, `STALE_VISIBLE`, and one action label, `Refresh`.

This is a prioritization baseline, not a prediction of causality. It uses only signals available at the decision moment.


In [4]:
# ============================================================
# Section 2 — Build the ranked baseline queue
# ============================================================

import os
import numpy as np
import pandas as pd

# Load the original starter dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# ------------------------------------------------------------
# Baseline rule
# ------------------------------------------------------------

# Signal 1: visible
visible = df["impressions_90d"] > 100

# Signal 2: meaningfully stale
stale = df["days_since_last_update"] >= 91

# A page is a refresh candidate only when BOTH signals are true
refresh_candidate = visible & stale

# ------------------------------------------------------------
# Score
# ------------------------------------------------------------

# Higher impressions = higher priority among qualifying pages.
df["baseline_score"] = np.where(
    refresh_candidate,
    df["impressions_90d"],
    0
)

# ------------------------------------------------------------
# One reason code
# ------------------------------------------------------------

df["reason_code"] = np.where(
    refresh_candidate,
    "STALE_VISIBLE",
    "NOT_SELECTED"
)

# ------------------------------------------------------------
# One action label
# ------------------------------------------------------------

df["action"] = np.where(
    refresh_candidate,
    "Refresh",
    "Monitor"
)

# ------------------------------------------------------------
# Rank the queue
# ------------------------------------------------------------

df["rank"] = (
    df["baseline_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------------------------------------
# Select the output columns
# ------------------------------------------------------------

output_cols = [
    "rank",
    "baseline_score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_direction"
]

baseline_queue = queue[output_cols].copy()

# ------------------------------------------------------------
# Show results
# ------------------------------------------------------------

print("Baseline queue created")
print("=" * 60)

print("Total pages:", len(baseline_queue))
print(
    "Refresh candidates:",
    (baseline_queue["action"] == "Refresh").sum()
)

print(
    "Monitor:",
    (baseline_queue["action"] == "Monitor").sum()
)

print("\nTop 10 baseline candidates:")
display(baseline_queue.head(10))

Baseline queue created
Total pages: 30000
Refresh candidates: 8115
Monitor: 21885

Top 10 baseline candidates:


,rank,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,1,517715,STALE_VISIBLE,Refresh,517715,104,4.2,0.14,down
1,2,443434,STALE_VISIBLE,Refresh,443434,104,27.9,0.21,stable
2,3,347399,STALE_VISIBLE,Refresh,347399,104,4.2,0.53,down
3,4,309910,STALE_VISIBLE,Refresh,309910,104,5.6,0.16,down
4,5,309192,STALE_VISIBLE,Refresh,309192,104,2.0,0.87,down
5,6,295097,STALE_VISIBLE,Refresh,295097,104,7.3,0.05,stable
6,7,286608,STALE_VISIBLE,Refresh,286608,104,26.2,0.06,stable
7,8,233561,STALE_VISIBLE,Refresh,233561,104,26.2,0.06,down
8,9,211366,STALE_VISIBLE,Refresh,211366,104,5.1,0.41,stable
9,10,208678,STALE_VISIBLE,Refresh,208678,104,9.7,0.00,down


In [6]:
# ============================================================
# Write ranked queue to the required output path
# ============================================================

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

baseline_queue.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows written:", len(baseline_queue))

Saved: work/outputs/baseline_action_score.csv
Rows written: 30000


In [7]:
# ============================================================
# Sanity checks
# ============================================================

refresh_rows = baseline_queue[
    baseline_queue["action"] == "Refresh"
]

print("Refresh candidates:", len(refresh_rows))

print(
    "All selected pages have >100 impressions:",
    (refresh_rows["impressions_90d"] > 100).all()
)

print(
    "All selected pages have >=91 days since update:",
    (refresh_rows["days_since_last_update"] >= 91).all()
)

print("\nReason codes:")
print(baseline_queue["reason_code"].value_counts())

print("\nActions:")
print(baseline_queue["action"].value_counts())

Refresh candidates: 8115
All selected pages have >100 impressions: True
All selected pages have >=91 days since update: True

Reason codes:
reason_code
NOT_SELECTED     21885
STALE_VISIBLE     8115
Name: count, dtype: int64

Actions:
action
Monitor    21885
Refresh     8115
Name: count, dtype: int64


In [8]:
import os

output_path = "work/outputs/baseline_action_score.csv"

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size:", os.path.getsize(output_path), "bytes")
    print("Path:", output_path)

File exists: True
File size: 1534872 bytes
Path: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# ============================================================
# Section 3 — Top-20 Review
# ============================================================

top20 = baseline_queue.head(20).copy()

top20_review = top20[
    [
        "rank",
        "baseline_score",
        "action",
        "reason_code",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr"
    ]
].copy()

display(top20_review)

,rank,baseline_score,action,reason_code,impressions_90d,days_since_last_update,avg_position,ctr
0,1,517715,Refresh,STALE_VISIBLE,517715,104,4.2,0.14
1,2,443434,Refresh,STALE_VISIBLE,443434,104,27.9,0.21
2,3,347399,Refresh,STALE_VISIBLE,347399,104,4.2,0.53
3,4,309910,Refresh,STALE_VISIBLE,309910,104,5.6,0.16
4,5,309192,Refresh,STALE_VISIBLE,309192,104,2.0,0.87
5,6,295097,Refresh,STALE_VISIBLE,295097,104,7.3,0.05
6,7,286608,Refresh,STALE_VISIBLE,286608,104,26.2,0.06
7,8,233561,Refresh,STALE_VISIBLE,233561,104,26.2,0.06
8,9,211366,Refresh,STALE_VISIBLE,211366,104,5.1,0.41
9,10,208678,Refresh,STALE_VISIBLE,208678,104,9.7,0.00


### Top-20 review

The rule selects pages that are both visible and stale. The score prioritizes pages with higher observed impressions.

For each selected page, I will review the action and reason code, note why the rule considers it a candidate, and identify what could make the recommendation wrong.

These are prioritization recommendations, not claims that a refresh will improve performance.


In [10]:
# ============================================================
# Section 3 — Top-20 Review
# ============================================================

top20_review = baseline_queue.head(20).copy()

# Confidence note based only on observed rule signals
top20_review["confidence_note"] = np.where(
    top20_review["impressions_90d"] >= 10000,
    "Strong visibility signal, but refresh need is not proven",
    "Visible and stale, but lower exposure than the strongest picks"
)

# What could make the recommendation wrong
top20_review["what_would_make_it_wrong"] = (
    "The page may still be performing well despite being stale, "
    "or the observed impressions may not justify a refresh."
)

review_cols = [
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20_review[review_cols])

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
1,2,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
2,3,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
3,4,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
4,5,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
5,6,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
6,7,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
7,8,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
8,9,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...
9,10,Refresh,STALE_VISIBLE,"Strong visibility signal, but refresh need is ...",The page may still be performing well despite ...


In [11]:
# Show the actual signals used by the rule
display(
    top20_review[
        [
            "rank",
            "baseline_score",
            "action",
            "reason_code",
            "impressions_90d",
            "days_since_last_update",
            "avg_position",
            "ctr"
        ]
    ]
)

,rank,baseline_score,action,reason_code,impressions_90d,days_since_last_update,avg_position,ctr
0,1,517715,Refresh,STALE_VISIBLE,517715,104,4.2,0.14
1,2,443434,Refresh,STALE_VISIBLE,443434,104,27.9,0.21
2,3,347399,Refresh,STALE_VISIBLE,347399,104,4.2,0.53
3,4,309910,Refresh,STALE_VISIBLE,309910,104,5.6,0.16
4,5,309192,Refresh,STALE_VISIBLE,309192,104,2.0,0.87
5,6,295097,Refresh,STALE_VISIBLE,295097,104,7.3,0.05
6,7,286608,Refresh,STALE_VISIBLE,286608,104,26.2,0.06
7,8,233561,Refresh,STALE_VISIBLE,233561,104,26.2,0.06
8,9,211366,Refresh,STALE_VISIBLE,211366,104,5.1,0.41
9,10,208678,Refresh,STALE_VISIBLE,208678,104,9.7,0.00


### Top-20 review

All top-20 pages satisfy the same baseline rule: they have more than 100 impressions in the last 90 days and have not been updated for at least 91 days.

The ranking is mainly driven by observed impressions, so higher-ranked pages have greater visibility. However, the rule does not prove that a refresh is needed or that refreshing the page will improve its performance.

A recommendation could be wrong if the page is already performing well, if its high impressions reflect a stable audience, or if updating the content would not address the actual reason for any performance change.


In [12]:
# ============================================================
# Section 3 — Top-20 Review
# ============================================================

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["impressions_90d"] >= 10000,
    "High visibility; refresh need is not proven",
    "Visible and stale; evidence is weaker than higher-volume pages"
)

top20["what_would_make_it_wrong"] = (
    "Page may already be performing well, or refresh may not address "
    "the reason for performance change"
)

top20_review = top20[
    [
        "rank",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
1,2,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
2,3,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
3,4,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
4,5,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
5,6,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
6,7,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
7,8,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
8,9,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."
9,10,Refresh,STALE_VISIBLE,High visibility; refresh need is not proven,"Page may already be performing well, or refres..."


### Weak picks and leakage check

Some picks are weaker than others because the baseline uses only two signals. A page can have high visibility and be stale without actually needing a refresh. For example, a page with a strong average position and high CTR may already be performing well.

The rule does not use the outcome label, trend direction, future-window data, or product decision flags. It only uses signals that are observable at the decision moment: impressions and days since last update.

Therefore, this baseline should be treated as a simple decision-support rule rather than proof that the selected pages will benefit from a refresh.


In [13]:
# ============================================================
# Section 4 — Leakage Check
# ============================================================

rule_features = [
    "impressions_90d",
    "days_since_last_update"
]

# Confirm the rule only depends on pre-decision signals
print("Features used by the baseline rule:")
print(rule_features)

# Check that outcome/product fields are NOT used
excluded_from_rule = [
    "is_declining",
    "trend_direction",
    "trend_pct",
    "future_impressions",
    "suggested_action",
    "final_refresh_score",
    "best_model_probability"
]

print("\nOutcome / decision fields intentionally excluded:")
for col in excluded_from_rule:
    print("-", col)

# Verify that the actual score is determined only by the two signals
expected_score = np.where(
    (df["impressions_90d"] > 100) &
    (df["days_since_last_update"] >= 91),
    df["impressions_90d"],
    0
)

print(
    "\nScore matches the two-signal rule:",
    np.array_equal(
        df["baseline_score"].values,
        expected_score
    )
)

Features used by the baseline rule:
['impressions_90d', 'days_since_last_update']

Outcome / decision fields intentionally excluded:
- is_declining
- trend_direction
- trend_pct
- future_impressions
- suggested_action
- final_refresh_score
- best_model_probability

Score matches the two-signal rule: True


In [14]:
# Pages with high visibility and stale status,
# but relatively strong current search performance.

weak_picks = baseline_queue[
    (baseline_queue["action"] == "Refresh") &
    (baseline_queue["avg_position"] <= 5) &
    (baseline_queue["ctr"] >= 0.50)
].head(10)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "rank",
            "baseline_score",
            "impressions_90d",
            "days_since_last_update",
            "avg_position",
            "ctr",
            "action",
            "reason_code"
        ]
    ]
)

Potential weak picks:


,rank,baseline_score,impressions_90d,days_since_last_update,avg_position,ctr,action,reason_code
2,3,347399,347399,104,4.2,0.53,Refresh,STALE_VISIBLE
4,5,309192,309192,104,2.0,0.87,Refresh,STALE_VISIBLE
20,21,174408,174408,104,4.7,0.56,Refresh,STALE_VISIBLE
34,35,143314,143314,104,1.9,0.83,Refresh,STALE_VISIBLE
36,37,142072,142072,104,3.6,0.83,Refresh,STALE_VISIBLE
41,42,131977,131977,104,4.4,0.72,Refresh,STALE_VISIBLE
42,43,131328,131328,104,2.4,0.70,Refresh,STALE_VISIBLE
48,49,129239,129239,104,3.6,0.55,Refresh,STALE_VISIBLE
57,58,125049,125049,104,4.5,0.86,Refresh,STALE_VISIBLE
61,62,117054,117054,104,4.4,0.57,Refresh,STALE_VISIBLE


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.